### Importing Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

#### Loading Dataset

In [2]:
torch.manual_seed(42)
np.random.seed(42)

housing = fetch_california_housing()
X_np, y_np = housing.data, housing.target
print(f'Features: {housing.feature_names}')
print(f'X: {X_np.shape}, y range: {y_np.min():.2f}, {y_np.max():.2f}')

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X: (20640, 8), y range: 0.15, 5.00


#### Splitting Data

In [3]:
X_temp, X_test, y_temp, y_test = train_test_split(X_np, y_np, test_size = 0.15, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.176, random_state = 42)

#### Scaling Features and Targets

In [4]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


y_mean, y_std = y_train.mean(), y_train.std()
y_train_s = (y_train - y_mean) / y_std
y_val_s = (y_val - y_mean) / y_std
y_test_s = (y_test - y_mean) / y_std

#### Custom Dataset

In [5]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = HousingDataset(X_train_s, y_train_s)
val_ds = HousingDataset(X_val_s, y_val_s)
test_ds = HousingDataset(X_test_s, y_test_s)


train_loader = DataLoader(train_ds, batch_size = 64, shuffle = True, num_workers = 0)
val_loader = DataLoader(val_ds, batch_size = 128, shuffle = False, num_workers = 0)
test_loader = DataLoader(test_ds, batch_size = 128, shuffle = False, num_workers = 0)

#### Inspecting

In [6]:
bX, by = next(iter(train_loader))
print(f"\nBatch X: {bX.shape}, mean={bX.mean():.3f}, std={bX.std():.3f}")
print(f"Batch y: {by.shape}")


Batch X: torch.Size([64, 8]), mean=-0.052, std=0.767
Batch y: torch.Size([64, 1])


#### Creating the Model

In [7]:
class HousingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1),   # regression — no activation
        )
    def forward(self, x): return self.net(x)

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = HousingNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#### Training and Validation

In [8]:
best_val_loss = float('inf')
for epoch in range(50):
    model.train()
    train_loss = 0.0

    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bX)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * bX.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bX, by in val_loader:
            bX, by = bX.to(device), by.to(device)
            val_loss += criterion(model(bX), by).item() * bX.size(0)
    val_loss /= len(val_ds)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_housing.pth')

    if (epoch + 1) % 10 == 0:
        train_rmse = np.sqrt(train_loss) * y_std * 100000
        val_rmse = np.sqrt(val_loss) * y_std * 100000
        print(f'Epoch {epoch + 1:>3} | Train RMSE: ${train_rmse:,.0f} | Val RMSE: ${val_rmse:,.0f}')

Epoch  10 | Train RMSE: $53,760 | Val RMSE: $54,411
Epoch  20 | Train RMSE: $51,397 | Val RMSE: $52,928
Epoch  30 | Train RMSE: $49,990 | Val RMSE: $52,255
Epoch  40 | Train RMSE: $48,498 | Val RMSE: $54,790
Epoch  50 | Train RMSE: $46,940 | Val RMSE: $53,331


#### Testing the Model

In [9]:
model.load_state_dict(torch.load('best_housing.pth'))
model.eval()
test_loss = 0.0
all_preds, all_true = [], []
with torch.no_grad():
    for bX, by in test_loader:
        bX, by = bX.to(device), by.to(device)
        pred = model(bX)
        test_loss += criterion(pred, by).item() * bX.size(0)
        all_preds.append(pred.cpu())
        all_true.append(by.cpu())

test_loss /= len(test_ds)
test_rmse = np.sqrt(test_loss) * y_std * 100000
print(f'Test RMSE: {test_rmse:,.0f}')

all_preds = torch.cat(all_preds).squeeze() * y_std + y_mean
all_true = torch.cat(all_true).squeeze() * y_std + y_mean

print('First 5 preds vs true (in $100K): ')
for i in range(5):
    print(f'Pred: {all_preds[i]:.3f} True: {all_true[i]:.3f}')

Test RMSE: 51,954
First 5 preds vs true (in $100K): 
Pred: 0.584 True: 0.477
Pred: 0.983 True: 0.458
Pred: 5.083 True: 5.000
Pred: 2.345 True: 2.186
Pred: 2.380 True: 2.780
